# Python for Neuroscience (2026)

# 02. Patch-clamp: analysis of action potentials

<img src="Images/patch-clamp_brain_slices.png" width="800">

Neurons encode and transfer information through action potentials (post). An action potential is a fast change of a few milliseconds in the resting membrane potential of cells, which for neurons usually ranges between -60 and -70 mV. The action potential consists of a rising phase, where the potential becomes less negative (**depolarization**) caused by the entry of Na+ ions, followed by a falling phase (**repolarization**) that usually goes below the initial resting potential due to a net efflux of K+.

<img src="Images/02_AP_Hodking-Huxley.png" width="400">


**Figure**. The first published intracellular recording of an action potential. Original figure and legend from the 1939 article by Hodgkin and Huxley: “Action Potentials Recorded from Inside a Nerve Fibre".

**GOAL**: How to analyze action potentials using the [IPFX library](https://github.com/AllenInstitute/ipfx) (Allen Institute for Brain Science) and custom methods.

**Further reading**
- Bean, 2007.The action potential in mammalian central neurons. [Link](https://neurophysics.ucsd.edu/courses/physics_171/Bean_AP_Review.pdf). *Must read if you want to know more about action potentials*.

# Import the libraries

**Important**: IPFX generally requires a separate environment with Python version 3.9 - 3.11. See [IPFX GitHub](https://github.com/alleninstitute/ipfx).

In [ ]:
!pip install pyabf ipfx

In [ ]:
# Import the packages
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# To import pClamp abf files into Python
import pyabf

# IPFX library for extract action potential features
from ipfx.feature_extractor import (SpikeFeatureExtractor,
                                    SpikeTrainFeatureExtractor)

# Function to detect action potentials
from scipy.signal import find_peaks

# To generate editable text in saved *.svg plots
plt.rcParams['svg.fonttype'] = 'none'

# Backend for interactive plots (optional)
# %matplotlib widget
# plt.close('all')

# Create the paths

**Data management: Resources**
- [FAIR Guiding Principles for scientific data management and stewardship](https://www.go-fair.org/fair-principles/)
- [NIN guidelines for data storage](https://nin.nl/wp-content/uploads/sites/2/2024/10/data-storage-protocol-NIN_20211210.pdf).
- [NIH data management](https://grants.nih.gov/policy-and-compliance/policy-topics/sharing-policies/dms/data-management).
- [Cambridge research data management](https://www.data.cam.ac.uk/about).

In [ ]:
# Create a folder name for the patch-clamp notebooks
notebook_name = 'patch-clamp'

# Current working directory (default is '/content' in Colab)
data_path = os.getcwd()

# Change the folder names accordingly
paths = {'data':  f'{data_path}/Example Data',
         'processed_data': f'{data_path}/Processed_data/{notebook_name}',
         'analysis': f'{data_path}/Analysis/{notebook_name}'}

# Make folders if they do not exist yet
for path in paths.values():
    os.makedirs(path, exist_ok=True)

# Download the data

In [ ]:
if 'google.colab' in str(get_ipython()):

    !wget -O "Example Data/pfc_pvalb_aps_01.abf" \
    "https://raw.githubusercontent.com/dav1dcg/neuro-notebook-templates/main/Python4Neuro_2026/Example%20data/pfc_pvalb_aps_01.abf"

    !wget -O "Example Data/pfc_pvalb_aps_02.abf" \
    "https://raw.githubusercontent.com/dav1dcg/neuro-notebook-templates/main/Python4Neuro_2026/Example%20data/pfc_pvalb_aps_02.abf"
else:
    print("Download data from 'Example Data' folder.")

## Q: Load the example data

<img src="Images/02_Pvalb_APs.png" width="500">

**Example data**: Whole-cell current-clamp recording from a parvalbumin interneuron in L2/3 of the mouse PFC. A series of depolarizing current steps were used to evoke action potentials.
- filename: **pfc_pvalb_aps_02.abf**

**Tips**:
- try/except syntax is useful for handling errors: https://docs.python.org/3/reference/compound_stmts.html#except
- if [os.path.exists()](https://docs.python.org/3/library/os.path.html#os.path.exists) s also useful to check if a data path exists and avoid errors.


In [ ]:
filename = "pfc_pvalb_aps_02"

try:
    data_path = f"{paths['data']}/{filename}.abf"
    abf = pyabf.ABF(data_path)
    print(abf)
except Exception as e:
    print(e)

<details>
<summary><strong>Solution</strong></summary>

```python

filename = "pfc_pvalb_aps_02"

try:
    data_path = f"{paths['data']}/{filename}.abf"
    abf = pyabf.ABF(data_path)
    print(abf)
except Exception as e:
    print(e)
```

# Plot the traces

## Quick plot

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, sharex=True)

voltage_channel = 0
current_channel = 0

for sweep in abf.sweepList:
    abf.setSweep(sweep, voltage_channel)
    ax1.plot(abf.sweepX, abf.sweepY, alpha=0.5)  # Alpha: transparency
    ax1.set_ylabel(abf.sweepLabelY)  # Label

    abf.setSweep(sweep, channel=current_channel)
    ax2.plot(abf.sweepX, abf.sweepC, color='black')
    ax2.set_ylabel(abf.sweepLabelC)
    ax2.set_xlabel(abf.sweepLabelX)
    ax2.set_xlim(0, 1)

plt.show()

## Q: Plot only one trace, customize labels, cleaner layout, and save the plot

**Tips**:
- `plt.rcParams`: https://matplotlib.org/stable/users/explain/customizing.html. Change default parameters in the script.
- `gridspec`: https://matplotlib.org/3.5.0/tutorials/intermediate/gridspec.html. Customize figure layouts.
- `ax.spines`: https://matplotlib.org/stable/api/spines_api.html
- `ax.tick_params`: https://matplotlib.org/stable/api/_as_gen/matplotlib.axes.Axes.tick_params.html
- `ax.set_xlim`: https://matplotlib.org/stable/api/_as_gen/matplotlib.axes.Axes.set_xlim.html
- `tight_layout`: https://matplotlib.org/stable/api/_as_gen/matplotlib.pyplot.tight_layout.html

In [ ]:

# Font type and size
plt.rcParams['font.size'] = 14

# Figure size and format
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(8, 4),
                               gridspec_kw={'height_ratios': [3, 1]}, sharex=True)

# Select channels and sweeps
voltage_channel = 0
current_channel = 0
sweep_number = 11

# Assuming `abf` is already loaded and prepared.
for sweep in abf.sweepList:

    # Plotting voltage
    abf.setSweep(sweep_number, voltage_channel)
    ax1.plot(abf.sweepX, abf.sweepY, color='red', linewidth=0.5)
    ax1.set_ylabel("V (mV)")

    # Customize ax1
    ax1.spines['top'].set_visible(False)
    ax1.spines['right'].set_visible(False)
    ax1.spines['bottom'].set_visible(False)
    ax1.spines['left'].set_position(('outward', 10))
    ax1.tick_params(bottom=False)  # hide ticks

    # Plotting current
    abf.setSweep(sweep_number, current_channel)
    ax2.plot(abf.sweepX, abf.sweepC, color='black', linewidth=0.5)
    ax2.set_ylabel("I (pA)")
    ax2.set_xlabel("Time (s)")
    ax2.set_xlim(0.2, 0.8)   # visible x-range

    # Customize spines for a floating effect
    ax2.spines['top'].set_visible(False)
    ax2.spines['right'].set_visible(False)
    ax2.spines['bottom'].set_visible(True)  # ensure bottom spine is visible
    ax2.spines['bottom'].set_position(('outward', 10))
    ax2.spines['left'].set_position(('outward', 10))

plt.tight_layout()
plt.show()

# Save the plot and the table
fig.savefig(f"{paths['analysis']}/{filename}_sweep{sweep_number}.svg", dpi=300)

<details>
<summary><strong>Solution</strong></summary>

```python

# Font type and size
plt.rcParams['font.family'] = 'Arial'
plt.rcParams['font.size'] = 14

# Figure size and format
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(8, 4),
                               gridspec_kw={'height_ratios': [3, 1]}, sharex=True)

# Select channels and sweeps
voltage_channel = 0
current_channel = 0
sweep_number = 11

# Assuming `abf` is already loaded and prepared.
for sweep in abf.sweepList:

    # Plotting voltage
    abf.setSweep(sweep_number, voltage_channel)
    ax1.plot(abf.sweepX, abf.sweepY, color='red', linewidth=0.5)
    ax1.set_ylabel("V (mV)")

    # Customize ax1
    ax1.spines['top'].set_visible(False)
    ax1.spines['right'].set_visible(False)
    ax1.spines['bottom'].set_visible(False)
    ax1.spines['left'].set_position(('outward', 10))
    ax1.tick_params(bottom=False)  # hide ticks

    # Plotting current
    abf.setSweep(sweep_number, current_channel)
    ax2.plot(abf.sweepX, abf.sweepC, color='black', linewidth=0.5)
    ax2.set_ylabel("I (pA)")
    ax2.set_xlabel("Time (s)")
    ax2.set_xlim(0.2, 0.8)   # visible x-range

    # Customize spines for a floating effect
    ax2.spines['top'].set_visible(False)
    ax2.spines['right'].set_visible(False)
    ax2.spines['bottom'].set_visible(True)  # ensure bottom spine is visible
    ax2.spines['bottom'].set_position(('outward', 10))
    ax2.spines['left'].set_position(('outward', 10))

plt.tight_layout()
plt.show()

# Save the plot and the table
fig.savefig(f"{paths['analysis']}/{filename}_sweep{sweep_number}.svg", dpi=300)
```

# Detect action potentials with numpy

Action potentials can be detected directly from voltage traces using NumPy by finding points that overpass a defined threshold. To avoid counting the same action potential twice, we can use time windows to identify "local maxima":
- [numpy.argmax](https://numpy.org/doc/stable/reference/generated/numpy.argmax.html). Returns the indices of the maximum values along an axis.

Empty lists (`[]`) can be used to store the indices and times of detected spikes while looping through the traces.

Detected spikes can be plotted as dots for visualization.

In [ ]:
# Assign variables
fs = abf.dataRate  # samples per second

# Set detection parameters
threshold = 0                     # Minimum voltage to consider a spike (mV)
time_window = int(0.005 * fs)      # in seconds

# Select one trace with action potentials
trace_no = 5

# Initialize lists to store spike info
spike_indices = []
spike_times = []

# Set the trace and variables
abf.setSweep(trace_no, voltage_channel)
time = abf.sweepX
voltage = abf.sweepY

i = 0
while i < len(voltage):
    if voltage[i] > threshold:
        # Look ahead in a short window to find the local maximum
        end_idx = min(i + time_window, len(voltage))
        # Find the voltage peak in that time window
        peak_index = i + np.argmax(voltage[i:end_idx])

        # APPEND PEAK INDEXES AND TIMES
        spike_indices.append(peak_index)
        spike_times.append(time[peak_index])

        # IMPORTANT: Skip forward to avoid counting the same spike twice
        i = peak_index + time_window
    else:
        i += 1

# Create a table with basic information
spikes_table = pd.DataFrame({
    'spike': np.arange(1, len(spike_indices)+1),
    'spike_index': spike_indices,
    'spike_time_s': spike_times})

# Plot the detected spikes
fig, ax = plt.subplots(figsize=(8,4))
ax.plot(time, voltage, label='Voltage trace')
ax.plot(np.array(spike_times), voltage[spike_indices], 'r.')
ax.set_xlabel("Time (s)")
ax.set_ylabel("Voltage (mV)")
ax.legend()
ax.set_xlim(0, 1)

plt.show()
spikes_table

# Detect action potentials with scipy

We can also use scipy [find_peaks](https://docs.scipy.org/doc/scipy/reference/generated/scipy.signal.find_peaks.html) to detect spikes by setting parameters like minimum threshold, prominence, and minimum distance between peaks. This function finds local maxima in a 1D signal array and returns the indices of the peaks that match your settings.

Empty tables (pd.DataFrame) will store the sweep number, spike count, indices, and times as we loop through each trace. Starting with an empty table can be useful because it lets you organize results into columns and add rows as you go.

Empty tables (`pd.DataFrame`) will store the sweep number, spike count, indices, and times as we loop through each trace. This follows the idea of Python as object-oriented programming: create an object first (dataframe in this case) that then you can modify or interact with it using its attributes and methods.

In [ ]:
# Set parameters for the Find peaks function (set to None if not needed)
thresh_min = 0                    # Min threshold to detect spikes
thresh_prominence = 50              # Min action potential height
distance_min = 1 * (fs/1000)        # Min horizontal distance between peaks (i.e. refractory period)

# Plot the detected spikes in the trace
fig, ax = plt.subplots(figsize=(8, 4))

# Create table with results
spikes_table = pd.DataFrame(columns = ['sweep', 'spike', 'spike_index', 'spike_time'])

for sweep in abf.sweepList[5:6]:
    abf.setSweep(sweep, 0)
    time = abf.sweepX
    peaks_signal = abf.sweepY

    # Find peaks function
    peaks, peaks_dict = find_peaks(peaks_signal,
               height=thresh_min,
               threshold=thresh_min,
               distance=distance_min,
               prominence=thresh_prominence,
               wlen=None,       # Window length to calculate prominence
               rel_height=0.5,  # Relative height at which the peak width is measured
               plateau_size=None)

    # plots traces and detected peaks
    ax.plot(time, peaks_signal)

    # Red dot on each detected spike
    ax.plot(peaks/fs, peaks_signal[peaks], "r.")

    # ANNOTATE THE SPIKES
    if peaks.size > 0:
        for j, peak_idx in enumerate(peaks):
            ax.annotate(j+1, (time[peak_idx], peaks_signal[peak_idx] + 1),
                        color='blue', fontsize=8)

        # sweep_table
        sweep_table = pd.DataFrame({
            'sweep': sweep,
            'spike': np.arange(1, len(peaks)+1),
            'spike_index': peaks,
            'spike_time': time[peaks]})

        # Concatenate tables from sweeps
        spikes_table = pd.concat([spikes_table, sweep_table], ignore_index=True)

# Final plot adjustments
ax.set_title("Event detection")
ax.set_xlabel("Time (s)")
ax.set_ylabel("Voltage (mV)")
ax.set_xlim(0, 1)

# Save results
fig.savefig(f"{paths['analysis']}/{filename}_scipy_APs.png", dpi=300)
spikes_table.to_csv(f"{paths['analysis']}/{filename}_scipy_APs.csv", index=False)

plt.show()
spikes_table

## Q: Calculate AP peak voltage and add to the table

**Tip**: See properties of find_peaks function (https://docs.scipy.org/doc/scipy/reference/generated/scipy.signal.find_peaks.html) to find which one can return the voltage value at each detected spike. Then, add it to the table.

In [ ]:
plt.rcParams["font.family"] = "DejaVu Sans"

# Set parameters for the Find peaks function (set to None if not needed)
thresh_min = 0                    # Min threshold to detect spikes
thresh_prominence = 15              # Min spike amplitude
distance_min = 1 * (fs/1000)        # Min horizontal distance between peaks

# Plot the detected spikes in the trace
fig, ax = plt.subplots(figsize=(8, 4))

# Create table with results
spikes_table = pd.DataFrame()

for sweep in abf.sweepList[5:6]:
    abf.setSweep(sweep, 0)
    time = abf.sweepX
    peaks_signal = abf.sweepY

    # Find peaks function
    peaks, peaks_dict = find_peaks(peaks_signal,
               height=thresh_min,
               threshold=thresh_min,
               distance=distance_min,
               prominence=thresh_prominence,
               wlen=None,       # Window length to calculate prominence
               rel_height=0.5,  # Relative height at which the peak width is measured
               plateau_size=None)

    # plots traces and detected peaks
    ax.plot(time, peaks_signal)

    # Red dot on each detected spike
    ax.plot(peaks/fs, peaks_signal[peaks], "r.")

    if peaks.size > 0:
        for j, peak_idx in enumerate(peaks):
            ax.annotate(j+1, (time[peak_idx], peaks_signal[peak_idx] + 1),
                        color='blue', fontsize=8)

        # sweep_table
        sweep_table = pd.DataFrame({
            'sweep': sweep,
            'AP': np.arange(1, len(peaks)+1),
            'AP_index': peaks,
            'AP_time': time[peaks],
            'AP_peak_voltage': peaks_dict['peak_heights']})  # NEW FEATURE

        # Concatenate tables from sweeps
        spikes_table = pd.concat([spikes_table, sweep_table], ignore_index=True)

# Final plot adjustments
ax.set_title("Event detection")
ax.set_xlabel("Time (s)")
ax.set_ylabel("Voltage (mV)")
ax.set_xlim(0, 1)

# Save results
fig.savefig(f"{paths['analysis']}/{filename}_scipy_APs.png", dpi=300)
spikes_table.to_csv(f"{paths['analysis']}/{filename}_scipy_APs.csv", index=False)

plt.show()
spikes_table

<details>
<summary><strong>Solution</strong></summary>

```python

# Set parameters for the Find peaks function (set to None if not needed)
thresh_min = 0                    # Min threshold to detect spikes
thresh_prominence = 15              # Min spike amplitude  
distance_min = 1 * (fs/1000)        # Min horizontal distance between peaks

# Plot the detected spikes in the trace
fig, ax = plt.subplots(figsize=(8, 4))

# Create table with results
spikes_table = pd.DataFrame()  

for sweep in abf.sweepList[5:6]:
    abf.setSweep(sweep, 0)  
    time = abf.sweepX
    peaks_signal = abf.sweepY
    
    # Find peaks function
    peaks, peaks_dict = find_peaks(peaks_signal,
               height=thresh_min,
               threshold=thresh_min,  
               distance=distance_min,  
               prominence=thresh_prominence,  
               wlen=None,       # Window length to calculate prominence
               rel_height=0.5,  # Relative height at which the peak width is measured
               plateau_size=None)
    
    # plots traces and detected peaks
    ax.plot(time, peaks_signal)

    # Red dot on each detected spike
    ax.plot(peaks/fs, peaks_signal[peaks], "r.")
    
    if peaks.size > 0:
        for j, peak_idx in enumerate(peaks):
            ax.annotate(j+1, (time[peak_idx], peaks_signal[peak_idx] + 1),
                        color='blue', fontsize=8)

        # sweep_table
        sweep_table = pd.DataFrame({
            'sweep': sweep,
            'AP': np.arange(1, len(peaks)+1),
            'AP_index': peaks,
            'AP_time': time[peaks],
            'AP_peak_voltage': peaks_dict['peak_heights']})  # NEW FEATURE
        
        # Concatenate tables from sweeps
        spikes_table = pd.concat([spikes_table, sweep_table], ignore_index=True)

# Final plot adjustments
ax.set_title("Event detection")
ax.set_xlabel("Time (s)")
ax.set_ylabel("Voltage (mV)")
ax.set_xlim(0, 1)

# Save results
fig.savefig(f"{paths['analysis']}/{filename}_scipy_APs.png", dpi=300)
spikes_table.to_csv(f"{paths['analysis']}/{filename}_scipy_APs.csv", index=False)

plt.show()
spikes_table

```

# Features of APs with IPFX

For the rest of the notebook, we will use [IPFX](https://github.com/AllenInstitute/ipfx), a package created by The Allen Institute for Brain Science to analyze intrinsic cell features from electrophysiological data. With this package you can:
- Detect action potentials and their features (e.g. threshold time and voltage).
- Calculate features of spike trains (e.g., adaptation index).
- Calculate stimulus-specific cell features.

<img src="Images/02_AP_features_Allen.png" width="500">

**Figure**. Anatomy of an action potential. Source: Electrophysiology white paper from Allen Institute for Brain Science.

**Single Action Potential Features**:
- **Action potential peak**: Maximum value of the membrane potential during the action potential.  
- **Action potential trough**: Minimum value of the membrane potential in the interval between the peak and the next action potential.  
- **Action potential full width**: Width of the action potential at half of its height.  
- **Action potential height**: Difference between the action potential peak and trough.  
- **Action potential voltage threshold**: Point where the rate of change of voltage (`dV/dt`) sharply increases, e.g., 5% of the average maximal `dV/dt`.

IPFX has a module called [`SpikeFeatureExtractor`](https://ipfx.readthedocs.io/en/latest/ipfx.html#ipfx.feature_extractor.SpikeFeatureExtractor) to get all features for each spike.

**To know more about the IPFX methods and functions**:
- [IPFX API](https://ipfx.readthedocs.io/en/latest/ipfx.html)

In [ ]:
# Define the analysis window (in seconds)
window_start = 0.2
window_end = 1

# Analysis parameters for action potential detection (use the ones you need)
voltage_threshold = 0  # Minimum absolute peak level in mV for spike detection
dv_cutoff = 20.0  # Minimum dV/dt to qualify as a spike in V/s
thresh_frac = 0.05  # fraction of average upstroke for threshold calculation

# Set the table from the dataframe below
dataframe = []

# Loop function to analyze each voltage trace of the file
for sweepNumber in abf.sweepList:
    abf.setSweep(sweepNumber)
    time = abf.sweepX
    voltage = abf.sweepY
    current = abf.sweepC

    # Define the region
    start, end = window_start, window_end

    # IPFX MODULE TO EXTRACT SPIKE FEATURES
    sfx = SpikeFeatureExtractor(start, end,
                                dv_cutoff=dv_cutoff,
                                thresh_frac=thresh_frac,
                                min_peak=voltage_threshold)
    sfx_results = sfx.process(time, voltage, current)
    dataframe.append(sfx_results)  # To get the mean: df.append(sfx_results.mean())

# Table with the features from all the action potentials
table = pd.concat(dataframe)

# Plot the trace/s
plt.figure(figsize=(6,4))
plt.xlabel ("Time (s)")
plt.ylabel("Voltage (mV)")

for sweepNumber in abf.sweepList:  # Loop to plot all the traces
    abf.setSweep(sweepNumber)
    plt.plot(abf.sweepX, abf.sweepY, alpha=.6, label="sweep %d" % (sweepNumber))

# Optional: To highlight one trace
abf.setSweep(4)
plt.plot(abf.sweepX, abf.sweepY, linewidth=1, color='black')
plt.xlim(window_start, window_end)

# Save the table and plot
plt.savefig(f"{paths['analysis']}/{filename}_spikefeatures_ipfx.svg", dpi=300)
table.to_csv(f"{paths['analysis']}/{filename}_spikefeatures_ipfx.csv", index=False)

# Display the graph and the table
plt.show()
table

# Remove the below # to show only selected columns. E.g:
columns = ['threshold_i', 'threshold_v', 'width', 'upstroke', 'downstroke']
table[columns]

## Q: Show only results of first AP from each trace and plot the peak index


<details>
<summary><strong>Tips</strong></summary>
    
- Print column headers with `table.columns`
- Filter the table with `first().reset_index()` and `peak_i`, then use `peak_index` for plotting


In [ ]:
# INSERT VALUES TO ZOOM IN
window_start =  0.24
window_end =  0.28

# Analysis parameters for action potential detection (use the ones you need)
voltage_threshold = 0  # Minimum absolute peak level in mV for spike detection
dv_cutoff = 20.0  # Minimum dV/dt to qualify as a spike in V/s
thresh_frac = 0.05  # fraction of average upstroke for threshold calculation

# Set the table from the dataframe below
dataframe = []

# Loop function to analyze each voltage trace of the file
for sweepNumber in abf.sweepList:
    abf.setSweep(sweepNumber)
    time = abf.sweepX
    voltage = abf.sweepY
    current = abf.sweepC

    # Define the region
    start, end = window_start, window_end

    # Parameters for analysis:
    sfx = SpikeFeatureExtractor (start, end,
                                 dv_cutoff=dv_cutoff,
                                 thresh_frac=thresh_frac,
                                 min_peak=voltage_threshold)
    sfx_results = sfx.process(time, voltage, current)
    dataframe.append(sfx_results)  # To get the mean: df.append(sfx_results.mean())

# Table with the features from all the action potentials
table = pd.concat(dataframe)

# FILTER FIRST VALUE OF EACH CURRENT STEP
# COMPLETE THE CODE
table = table.groupby('threshold_i').first().reset_index()

# Plot the trace/s
plt.figure(figsize=(6,4))
plt.xlabel ("Time (s)")
plt.ylabel("Voltage (mV)")
plt.xlim(window_start, window_end)

# COLOR MAP FOR MARKERS
colors = plt.cm.tab10(np.linspace(0, 1, len(abf.sweepList)))

for i, sweepNumber in enumerate(abf.sweepList):  # enumerate gives index i
    abf.setSweep(sweepNumber)
    time = abf.sweepX
    voltage = abf.sweepY
    current_step = np.max(abf.sweepC)

    # PLOT SWEEPS IN GRAY
    plt.plot(time, voltage, color='gray', alpha=0.5)

    # MARK THE FIRST AP INDEX FOR EACH SWEEP
    spikes_in_current_step = table[table['threshold_i'] == current_step]
    if not spikes_in_current_step.empty:
        # FIST PEAK INDEX IN SWEEP
        # Add the marker with the label as threshold_i


        idx = int(spikes_in_current_step['peak_index'].values[0])
        # Add the marker with the label as threshold_i
        plt.plot(time[idx], voltage[idx], 'o', color=colors[i], markersize=6,
                 label=f"{current_step} pA")  # label

# After the loop, show the legend
plt.legend(title="Current step", loc='upper right') # COMPLETE THE CODE)

# Save the table and plot
plt.savefig(f"{paths['analysis']}/{filename}_ipfx_AP1.svg", dpi=300)
table.to_csv(f"{paths['analysis']}/{filename}_ipfx_AP1.csv", index=False)

# Display the graph and the table
plt.show()
table

<details>
<summary><strong>Solution</strong></summary>

```python
# ZOOM IN
window_start = 0.245
window_end = 0.28

# Analysis parameters for action potential detection (use the ones you need)
voltage_threshold = 0  # Minimum absolute peak level in mV for spike detection
dv_cutoff = 20.0  # Minimum dV/dt to qualify as a spike in V/s
thresh_frac = 0.05  # fraction of average upstroke for threshold calculation

# Set the table from the dataframe below
dataframe = []

# Loop function to analyze each voltage trace of the file
for sweepNumber in abf.sweepList:
    abf.setSweep(sweepNumber)
    time = abf.sweepX
    voltage = abf.sweepY
    current = abf.sweepC
    
    # Define the region
    start, end = window_start, window_end
    
    # Parameters for analysis:
    sfx = SpikeFeatureExtractor (start, end,
                                 dv_cutoff=dv_cutoff,  
                                 thresh_frac=thresh_frac,  
                                 min_peak=voltage_threshold)  
    sfx_results = sfx.process(time, voltage, current)
    dataframe.append(sfx_results)  # To get the mean: df.append(sfx_results.mean())
    
# Table with the features from all the action potentials
table = pd.concat(dataframe)

# FILTER FIRST VALUE OF EACH CURRENT STEP
table = table.groupby('threshold_i').first().reset_index()

# Plot the trace/s
plt.figure(figsize=(6,4))
plt.xlabel ("Time (s)")
plt.ylabel("Voltage (mV)")
plt.xlim(window_start, window_end)

# COLOR MAP FOR MARKERS
colors = plt.cm.tab10(np.linspace(0, 1, len(abf.sweepList)))

for i, sweepNumber in enumerate(abf.sweepList):  # enumerate gives index i
    abf.setSweep(sweepNumber)
    time = abf.sweepX
    voltage = abf.sweepY
    current_step = np.max(abf.sweepC)
    
    # PLOT SWEEPS IN GRAY
    plt.plot(time, voltage, color='gray', alpha=0.5)
    
    # MARK THE FIRST AP INDEX FOR EACH SWEEP
    spikes_in_current_step = table[table['threshold_i'] == current_step]
    if not spikes_in_current_step.empty:
        idx = int(spikes_in_current_step['peak_index'].values[0])
        # Add the marker with the label as threshold_i
        plt.plot(time[idx], voltage[idx], 'o', color=colors[i], markersize=6,
                 label=f"{current_step} pA")  # label

# After the loop, show the legend
plt.legend(title="Current step", loc='upper right')                    

# Save the table and plot
plt.savefig(f"{paths['analysis']}/{filename}_ipfx_AP1.svg", dpi=300)
table.to_csv(f"{paths['analysis']}/{filename}_ipfx_AP1.csv", index=False)
   
# Display the graph and the table
plt.show()
table

```

# Spike train features with IPFX

<img src="Images/02_Firing-patterns_Bear_2015.png" width="1000">

**Figure**. Different firing patterns of cortical neurons: Fast-spiking (inhibitory), and adapting and bursting firing (pyramidal neurons). Adapted from Neuroscience: Exploring the brain, 4th (2015).

In addition to single action potential features, several features were calculated based on trains of action potentials (see Figure below).

<img src="Images/02_Firing-features_IPFX.png" width="500">

**Figure**. Sweep illustration of features such as latency, first ISI, and other ISIs. More complex features are calculated based on combinations of these features. Source: Allen Institute for Brain Science.

**Action potential train features**
- First ISI: The ISI between the first two spikes evoked by a stimulus.
- Average ISI: The mean value of all interspike intervals in a sweep.
- ISI coefficient of variation: The coefficient of variation (CV, or standard deviation divided by the mean) of all interspike intervals in a sweep.
- Average firing rate: The number of spikes during a stimulus period divided by the duration of the stimulus period.
- Latency: Time between the start of the stimulus and the first spike evoked by the stimulus.
- Adaptation index: The rate at which firing speeds up or slows down during a stimulus.

**IMPORTANT**: IPFX calculates the firing rate (spikes/sec) instead of the number of action potentials. To get the number of action potentials, either select a 1-second analysis window or multiply the 'avg_rate' by the time window width (e.g., 'avg_rate' * 0.6 s).

In [ ]:
# Define the analysis window in seconds
window_start = 0.1
window_end = 1.1
highlighted_trace = 6

# Define the region detection parameters
voltage_threshold = 0  # Minimum acceptable absolute peak level in mV
dv_cutoff = 20.0  # Minimum dV/dt to qualify as a spike in V/s
thresh_frac = 0.05  # Fraction of average upstroke for threshold calculation

# Define the current average segment in ms (or select just one timepoint)
current_t1 = 400
current_t2 = 500

# Define the channels (0 by default)
voltage_channel = 0
current_channel = 0

# Set the dataframe for the table
table = pd.DataFrame()

# Example with two subplots
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(8, 5), sharex=True)

# Loop function to analyze each voltage and current trace of the file
for sweep in abf.sweepList:

    # if sweep != highlighted_trace:  # Uncomment to show results from one trace only
    #     continue

    abf.setSweep(sweep, channel=voltage_channel)
    time = abf.sweepX
    voltage = abf.sweepY

    abf.setSweep(sweep, channel=current_channel)
    current = abf.sweepC

    # Current value between t1 and t2 (ms) for each step
    t1_idx = int(current_t1 * abf.dataPointsPerMs)
    t2_idx = int(current_t2 * abf.dataPointsPerMs)
    current_mean = np.average(current[t1_idx:t2_idx])

    # Define the region (in sec) and detection parameters
    start, end = window_start, window_end

    # Run the function to extract spike features
    sfx = SpikeFeatureExtractor(
        start=start, end=end,
        filter=None,
        dv_cutoff=dv_cutoff,
        thresh_frac=thresh_frac,
        min_peak=voltage_threshold)

    sfx_results = sfx.process(time, voltage, current)

    stfx = SpikeTrainFeatureExtractor(start, end)
    stfx_results = stfx.process(time, voltage, current, sfx_results)

    # Create a table with the stfx results
    length = len(table)
    table.loc[length, 'current_step'] = current_mean
    table.loc[length, 'avg_rate'] = stfx_results["avg_rate"]
    if stfx_results['avg_rate'] > 0:
        table.loc[length, 'adaptation'] = stfx_results["adapt"]
        table.loc[length, 'Latency_s'] = stfx_results["latency"]
        table.loc[length, 'ISI_CV'] = stfx_results["isi_cv"]
        table.loc[length, 'ISI_mean_ms'] = stfx_results["mean_isi"]*1000
        table.loc[length, 'ISI_first_ms'] = stfx_results["first_isi"]*1000

        # Show detection of action potential peak
        ax1.plot(sfx_results["peak_t"], sfx_results["peak_v"], 'r.', label='Peak')
        ax1.plot(sfx_results["threshold_t"], sfx_results["threshold_v"], 'k.', label='Threshold')

    # All voltage traces
    abf.setSweep(sweep, channel=voltage_channel)
    ax1.plot(abf.sweepX, abf.sweepY, alpha=.6, label=f"Sweep {sweep}")

    # Currents
    abf.setSweep(sweep, channel=current_channel)
    ax2.plot(abf.sweepX, abf.sweepC)

# Voltage traces
ax1.set_ylabel('Membrane voltage (mV)')
ax1.axes.set_xlim(window_start - 0.1, window_end + 0.1)
# ax1.legend()

# Plot the voltage threshold and time window lines
ax1.axhline(voltage_threshold, color='gray', linestyle='--')
ax1.axvline(start, linestyle="dotted", color="gray")
ax1.axvline(end, linestyle="dotted", color="gray")

# Plot the current steps
ax2.set_ylabel("Current (pA)")
ax2.set_xlabel("Time (s)")

# Highlight one current trace
abf.setSweep(highlighted_trace, channel=current_channel)
ax2.plot(abf.sweepX, abf.sweepC, linewidth=1, color='black')

# Tight layout
fig.tight_layout()

# Calculate the rheobase
rheobase_index = table[table['avg_rate'] > 0].index[0]
rheobase = table.loc[rheobase_index, 'current_step']
print("Rheobase (pA):", rheobase)

# Save the table and plot
plt.savefig(f"{paths['analysis']}/{filename}_ipfx_APs.svg", dpi=300)
table.to_csv(f"{paths['analysis']}/{filename}_ipfx_APs.csv", index=False)

# Show the graph and the table results
plt.show()
table

## Q: Create an Input-output (number of APs vs current steps) plot


<img src="Images/02_APs_Input-output_IPFX.png" width="500">

**Figure**. Input-output curve shows the number of action potentials (or firing frequency) in response to each current step.

**Tips**:
- Use [`ax.scatter`](https://matplotlib.org/stable/api/_as_gen/matplotlib.axes.Axes.scatter.html).

In [ ]:
# Define the analysis window in seconds
window_start = 0.1
window_end = 1.1
highlighted_trace = 6

# Define the region detection parameters
voltage_threshold = 0  # Minimum acceptable absolute peak level in mV
dv_cutoff = 20.0  # Minimum dV/dt to qualify as a spike in V/s
thresh_frac = 0.05  # Fraction of average upstroke for threshold calculation

# Define the current average segment in ms
current_t1 = 400
current_t2 = 500

# Define the channels (0 by default)
voltage_channel = 0
current_channel = 0

# Set the dataframe for the table
table = pd.DataFrame()

# Example with two subplots
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(5, 5), sharex=True)

# Loop function to analyze each voltage and current trace of the file
for sweep in abf.sweepList:

    # if sweep != highlighted_trace:  # Uncomment to show results from one trace only
    #     continue

    abf.setSweep(sweep, channel=voltage_channel)
    time = abf.sweepX
    voltage = abf.sweepY

    abf.setSweep(sweep, channel=current_channel)
    current = abf.sweepC

    # Current value between t1 and t2 (ms) for each step
    t1_idx = int(current_t1 * abf.dataPointsPerMs)
    t2_idx = int(current_t2 * abf.dataPointsPerMs)
    current_mean = np.average(current[t1_idx:t2_idx])

    # Define the region (in sec) and detection parameters
    start, end = window_start, window_end

    # Run the function to extract spike features
    sfx = SpikeFeatureExtractor(
        start=start, end=end,
        filter=None,
        dv_cutoff=dv_cutoff,
        thresh_frac=thresh_frac,
        min_peak=voltage_threshold)

    sfx_results = sfx.process(time, voltage, current)

    stfx = SpikeTrainFeatureExtractor(start, end)
    stfx_results = stfx.process(time, voltage, current, sfx_results)

    # Create a table with the stfx results
    length = len(table)
    table.loc[length, 'current_step'] = current_mean
    table.loc[length, 'avg_rate'] = stfx_results["avg_rate"]
    if stfx_results['avg_rate'] > 0:
        table.loc[length, 'adaptation'] = stfx_results["adapt"]
        table.loc[length, 'Latency_s'] = stfx_results["latency"]
        table.loc[length, 'ISI_CV'] = stfx_results["isi_cv"]
        table.loc[length, 'ISI_mean_ms'] = stfx_results["mean_isi"]*1000
        table.loc[length, 'ISI_first_ms'] = stfx_results["first_isi"]*1000

        # Show detection of action potential peak
        ax1.plot(sfx_results["peak_t"], sfx_results["peak_v"], 'r.', label='Peak')
        ax1.plot(sfx_results["threshold_t"], sfx_results["threshold_v"], 'k.', label='Threshold')

    # All voltage traces
    abf.setSweep(sweep, channel=voltage_channel)
    ax1.plot(abf.sweepX, abf.sweepY, alpha=.6, label=f"Sweep {sweep}")

    # Currents
    abf.setSweep(sweep, channel=current_channel)
    ax2.plot(abf.sweepX, abf.sweepC)

# Voltage traces
ax1.set_ylabel('Membrane voltage (mV)')
ax1.axes.set_xlim(window_start - 0.1, window_end + 0.1)
# ax1.legend()

# Plot the voltage threshold and time window lines
ax1.axhline(voltage_threshold, color='gray', linestyle='--')
ax1.axvline(start, linestyle="dotted", color="gray")
ax1.axvline(end, linestyle="dotted", color="gray")

# Plot the current steps
ax2.set_ylabel("Current (pA)")
ax2.set_xlabel("Time (s)")

# Highlight one current trace
abf.setSweep(highlighted_trace, channel=current_channel)
ax2.plot(abf.sweepX, abf.sweepC, linewidth=1, color='black')

# Tight layout
fig.tight_layout()

# Calculate the rheobase
rheobase_index = table[table['avg_rate'] > 0].index[0]
rheobase = table.loc[rheobase_index, 'current_step']
print("Rheobase (pA):", rheobase)

# Save the table and plot
plt.savefig(f"{paths['analysis']}/{filename}_ipfx_APs.svg", dpi=300)
table.to_csv(f"{paths['analysis']}/{filename}_ipfx_APs.csv", index=False)

# NEW: IV PLOT
# ax_iv.plots # INSERT LINE TO CREATE PLOT

# Get currents for all sweeps
currents = []
# COMPLETE THE CODE

# Scatter plot: Current step vs number of APs
# COMPLETE THE CODE

# Show figure
plt.tight_layout()
plt.show()

# Show the graph and the table results
plt.show()
table

<details>

<summary><strong>Solution</strong></summary>

```python

# Define the analysis window in seconds
window_start = 0.1
window_end = 1.1
highlighted_trace = 6

# Define the region detection parameters
voltage_threshold = 0  # Minimum acceptable absolute peak level in mV
dv_cutoff = 20.0  # Minimum dV/dt to qualify as a spike in V/s
thresh_frac = 0.05  # Fraction of average upstroke for threshold calculation

# Define the current average segment in ms
current_t1 = 400
current_t2 = 500

# Define the channels (0 by default)
voltage_channel = 0
current_channel = 0

# Set the dataframe for the table
table = pd.DataFrame()

# Example with two subplots
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(5, 5), sharex=True)

# Loop function to analyze each voltage and current trace of the file
for sweep in abf.sweepList:

    # if sweep != highlighted_trace:  # Uncomment to show results from one trace only
    #     continue  
        
    abf.setSweep(sweep, channel=voltage_channel)
    time = abf.sweepX
    voltage = abf.sweepY
    
    abf.setSweep(sweep, channel=current_channel)
    current = abf.sweepC
    
    # Current value between t1 and t2 (ms) for each step
    t1_idx = int(current_t1 * abf.dataPointsPerMs)
    t2_idx = int(current_t2 * abf.dataPointsPerMs)
    current_mean = np.average(current[t1_idx:t2_idx])
    
    # Define the region (in sec) and detection parameters
    start, end = window_start, window_end
    
    # Run the function to extract spike features
    sfx = SpikeFeatureExtractor(
        start=start, end=end,
        filter=None,  
        dv_cutoff=dv_cutoff,
        thresh_frac=thresh_frac,
        min_peak=voltage_threshold)
    
    sfx_results = sfx.process(time, voltage, current)
    
    stfx = SpikeTrainFeatureExtractor(start, end)
    stfx_results = stfx.process(time, voltage, current, sfx_results)
    
    # Create a table with the stfx results
    length = len(table)
    table.loc[length, 'current_step'] = current_mean
    table.loc[length, 'avg_rate'] = stfx_results["avg_rate"]
    if stfx_results['avg_rate'] > 0:
        table.loc[length, 'adaptation'] = stfx_results["adapt"]
        table.loc[length, 'Latency_s'] = stfx_results["latency"]
        table.loc[length, 'ISI_CV'] = stfx_results["isi_cv"]
        table.loc[length, 'ISI_mean_ms'] = stfx_results["mean_isi"]*1000
        table.loc[length, 'ISI_first_ms'] = stfx_results["first_isi"]*1000
        
        # Show detection of action potential peak
        ax1.plot(sfx_results["peak_t"], sfx_results["peak_v"], 'r.', label='Peak')
        ax1.plot(sfx_results["threshold_t"], sfx_results["threshold_v"], 'k.', label='Threshold')
        
    # All voltage traces
    abf.setSweep(sweep, channel=voltage_channel)
    ax1.plot(abf.sweepX, abf.sweepY, alpha=.6, label=f"Sweep {sweep}")
        
    # Currents
    abf.setSweep(sweep, channel=current_channel)
    ax2.plot(abf.sweepX, abf.sweepC)
     
# Voltage traces
ax1.set_ylabel('Membrane voltage (mV)')
ax1.axes.set_xlim(window_start - 0.1, window_end + 0.1)
# ax1.legend()

# Plot the voltage threshold and time window lines
ax1.axhline(voltage_threshold, color='gray', linestyle='--')
ax1.axvline(start, linestyle="dotted", color="gray")
ax1.axvline(end, linestyle="dotted", color="gray")

# Plot the current steps
ax2.set_ylabel("Current (pA)")
ax2.set_xlabel("Time (s)")

# Highlight one current trace
abf.setSweep(highlighted_trace, channel=current_channel)
ax2.plot(abf.sweepX, abf.sweepC, linewidth=1, color='black')

# Tight layout
fig.tight_layout()

# Calculate the rheobase
rheobase_index = table[table['avg_rate'] > 0].index[0]
rheobase = table.loc[rheobase_index, 'current_step']
print("Rheobase (pA):", rheobase)

# Save the table and plot
plt.savefig(f"{paths['analysis']}/{filename}_ipfx_APs.svg", dpi=300)
table.to_csv(f"{paths['analysis']}/{filename}_ipfx_APs.csv", index=False)

# NEW: IV PLOT
fig_iv, ax_iv = plt.subplots(figsize=(3,3))

# Get currents for all sweeps
currents = []
for sweep in abf.sweepList:
    abf.setSweep(sweep, channel=current_channel)
    currents.append(np.average(abf.sweepC[t1_idx:t2_idx]))

# Scatter plot: Current step vs number of APs
ax_iv.scatter(currents, table['avg_rate'], color='blue')
ax_iv.set_xlabel('Current step (pA)')
ax_iv.set_ylabel('APs')

# Show figure
plt.tight_layout()
plt.show()

# Show the graph and the table results
plt.show()
table
```

## Q: How to get a table with results from several recordings?

**Example data**: `pfc_pvalb_aps_01.abf`, `pfc_pvalb_aps_02.abf`

**Tips**:

- Use **loops** to process multiple files or sweeps efficiently. One loop can go through each file, and another can go through each sweep within the file.
- Use [os.listdir](https://docs.python.org/3/library/os.html#os.listdir) to get all files in a folder.  
- Filter recordings by name using [`str.startswith`](https://docs.python.org/3/library/stdtypes.html#str.startswith) to select only the relevant files, such as those starting with `"pfc_pvalb_aps"`. This is one of the reasons why having a consistent file naming convention ([examples](https://datamanagement.hms.harvard.edu/plan-design/file-naming-conventions)) is very useful. See also [FAIR principles](https://www.go-fair.org/fair-principles/) for scientific data management and stewardship.


In [ ]:
# Set the dataframe for the table
table_all = pd.DataFrame()

abf = pyabf.ABF(abf_path)

# INSERT THE LOOP
# Paste below the code you want to apply to the files
for sweep in abf.sweepList:
    abf.setSweep(sweep)
    time = abf.sweepX
    voltage = abf.sweepY
    current = abf.sweepC

    currents = [] # Current value between t1 and t2 (ms) for each step
    t1 = int(400*abf.dataPointsPerMs)
    t2 = int(500*abf.dataPointsPerMs)
    current_mean = np.average(abf.sweepC[t1:t2])

    start, end = 0.1, 1.1
    sfx = SpikeFeatureExtractor(start=start, end=end, filter=None)
    sfx_results = sfx.process(time, voltage, current)
    stfx = SpikeTrainFeatureExtractor (start=start, end=end)
    stfx_results = stfx.process(time, voltage, current, sfx_results)

    # Create a table with the stfx results
    length = len(table_all)
    table_all.loc[length, 'file_name'] = filename
    table_all.loc[length, 'current_step'] = current_mean
    table_all.loc[length, 'avg_rate'] = stfx_results["avg_rate"]
    if stfx_results ['avg_rate'] > 0:
        table_all.loc[length, 'adaptation'] = stfx_results ["adapt"]
        table_all.loc[length, 'Latency_s'] = stfx_results ["latency"]
        table_all.loc[length, 'ISI_CV'] = stfx_results ["isi_cv"]
        table_all.loc[length, 'ISI_mean_ms'] = stfx_results ["mean_isi"]*1000
        table_all.loc[length, 'ISI_first_ms'] = stfx_results ["first_isi"]*1000

# Save the table
table.to_csv(f"{paths['analysis']}/{experiment}_ipfx_APs_all.csv", index=False)

# Show the table (optional)
table_all

<details>

<summary><strong>Solution</strong></summary>

```python

# Set the dataframe for the table
table_all = pd.DataFrame()

experiment = 'pfc_pvalb_aps'

# Another option is:
# for root, dirs, files in os.walk(paths['data']):
#     for file in files:

for file in os.listdir(paths['data']):
    # Filter files by name and extension
    if file.startswith(experiment) and file.endswith('.abf'):
        abf_path = os.path.join(paths['data'], file)
        abf = pyabf.ABF(abf_path)

        # Get the filename without the extension
        filename = os.path.splitext(file)[0]
        
        # Paste below the code you want to apply to the files
        for sweep in abf.sweepList:
            abf.setSweep(sweep)
            time = abf.sweepX
            voltage = abf.sweepY
            current = abf.sweepC

            currents = [] # Current value between t1 and t2 (ms) for each step
            t1 = int(400*abf.dataPointsPerMs)
            t2 = int(500*abf.dataPointsPerMs)
            current_mean = np.average(abf.sweepC[t1:t2])

            start, end = 0.1, 1.1
            sfx = SpikeFeatureExtractor(start=start, end=end, filter=None)
            sfx_results = sfx.process(time, voltage, current)
            stfx = SpikeTrainFeatureExtractor (start=start, end=end)
            stfx_results = stfx.process(time, voltage, current, sfx_results)

            # Create a table with the stfx results
            length = len(table_all)
            table_all.loc[length, 'file_name'] = filename
            table_all.loc[length, 'current_step'] = current_mean
            table_all.loc[length, 'avg_rate'] = stfx_results["avg_rate"]
            if stfx_results ['avg_rate'] > 0:
                table_all.loc[length, 'adaptation'] = stfx_results ["adapt"]
                table_all.loc[length, 'Latency_s'] = stfx_results ["latency"]
                table_all.loc[length, 'ISI_CV'] = stfx_results ["isi_cv"]
                table_all.loc[length, 'ISI_mean_ms'] = stfx_results ["mean_isi"]*1000
                table_all.loc[length, 'ISI_first_ms'] = stfx_results ["first_isi"]*1000

# Save the table
table.to_csv(f"{paths['analysis']}/{experiment}_ipfx_APs_all.csv", index=False)

# Show the table (optional)
table_all
```